# Defaqto Sales DB — 01 · Explore the data, then build the pipeline

**Author:** Ketki Kothe · Solution Engineer, Snowflake · `Snowflake Solution Engineering`
**Workshop:** Snowflake × Defaqto, Tuesday 8 September 2026, Snowflake London
**Aggregation logic:** supplied by the Defaqto data lead

Three parts:

1. **Look at what landed** — seven tables, one cell each.
2. **Find the thing nobody has looked at** — one column explains the whole funnel.
3. **Build ten dynamic tables** — seven silver, three gold, each with one sentence
   saying what you get from it.

Everything you build lands in **your own schema**. Set your alias in the next cell
and nothing you do can collide with anybody else in the room.

> `DEFAQTO_DB.RAW` is shared and read-only. This notebook never writes to it.

## Set your alias

Change `kkothe` to your own alias — surname or initials, lowercase, no spaces.
Everything else in the notebook follows from this one cell.

In [ ]:
%%sql -r dataframe_1
-- CHANGE THIS ONE LINE, then run every cell below unchanged.
SET alias = 'kkothe';

SET my_schema = 'DEFAQTO_DB.TRANSFORMED_' || UPPER($alias);

USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
CREATE SCHEMA IF NOT EXISTS IDENTIFIER($my_schema);
USE SCHEMA IDENTIFIER($my_schema);

SELECT CURRENT_SCHEMA() AS you_are_working_in, CURRENT_WAREHOUSE() AS warehouse;

## 1 · What actually landed

Seven tables. Six describe the quote journey on short-term car; the seventh is the
sales feed covering all ten product types.

In [ ]:
%%sql -r dataframe_2
-- The shape of the whole dataset in one result.
SELECT 'STC_QUOTES'              AS table_name, COUNT(*) AS n_rows FROM DEFAQTO_DB.RAW.STC_QUOTES
UNION ALL SELECT 'STC_RATES',               COUNT(*) FROM DEFAQTO_DB.RAW.STC_RATES
UNION ALL SELECT 'STC_CLICKS',              COUNT(*) FROM DEFAQTO_DB.RAW.STC_CLICKS
UNION ALL SELECT 'STC_PERSONAL_ATTRIBUTES', COUNT(*) FROM DEFAQTO_DB.RAW.STC_PERSONAL_ATTRIBUTES
UNION ALL SELECT 'STC_COVERDETAILS',        COUNT(*) FROM DEFAQTO_DB.RAW.STC_COVERDETAILS
UNION ALL SELECT 'STC_VEHICLES',            COUNT(*) FROM DEFAQTO_DB.RAW.STC_VEHICLES
UNION ALL SELECT 'SALESDB_SALESEVENTS',     COUNT(*) FROM DEFAQTO_DB.RAW.SALESDB_SALESEVENTS
ORDER BY n_rows DESC;

### Look at each table

One cell per table. Skim the columns — you are looking for what is missing as much
as what is there.

In [ ]:
%%sql -r dataframe_3
-- One row per comparison journey. Note VISITED_RESULTS: we come back to it.
-- SOURCE and S1 are always NULL, and AFFILIATE_ID has no name column anywhere.
SELECT * FROM DEFAQTO_DB.RAW.STC_QUOTES LIMIT 20;

In [ ]:
%%sql -r dataframe_4
-- Every price a provider returned. STATUS 1 = priced, 3 = declined.
-- Declines still carry PRICE 0, so AVG(PRICE) without a filter is wrong.
SELECT * FROM DEFAQTO_DB.RAW.STC_RATES LIMIT 20;

In [ ]:
%%sql -r dataframe_5
-- Click-outs. Each resolves to a specific RATE_ID, so this records WHICH price
-- was chosen, not merely which provider. VALID = 0 means a duplicate click.
SELECT * FROM DEFAQTO_DB.RAW.STC_CLICKS LIMIT 20;

In [ ]:
%%sql -r dataframe_6
-- Age, outward postcode, occupation. This is the data Ben says never reaches
-- a sale. It exists - it is just never joined.
SELECT * FROM DEFAQTO_DB.RAW.STC_PERSONAL_ATTRIBUTES LIMIT 20;

In [ ]:
%%sql -r dataframe_7
-- Cover reason and length. Some rows carry a unit with no value.
SELECT * FROM DEFAQTO_DB.RAW.STC_COVERDETAILS LIMIT 20;

In [ ]:
%%sql -r dataframe_8
-- Make, model, year, value. VEHICLE_MODEL_YEAR is what answers the cohort question
-- about cars under seven years old.
SELECT * FROM DEFAQTO_DB.RAW.STC_VEHICLES LIMIT 20;

In [ ]:
%%sql -r dataframe_9
-- The sales feed: all ten product types, no primary key, and QUOTE_ID is unique
-- only WITHIN a product type. Ignore that and you double-count.
SELECT * FROM DEFAQTO_DB.RAW.SALESDB_SALESEVENTS LIMIT 20;

In [ ]:
%%sql -r dataframe_10
-- Before building anything: how much of this data can actually be analysed?
-- The three attribute tables nest, because one journey fills all three.
SELECT
    COUNT(*)                                                       AS quotes,
    COUNT(p.QUOTE_ID)                                              AS has_personal,
    COUNT(c.QUOTE_ID)                                              AS has_cover,
    COUNT(v.QUOTE_ID)                                              AS has_vehicle,
    COUNT_IF(p.QUOTE_ID IS NOT NULL
         AND c.QUOTE_ID IS NOT NULL
         AND v.QUOTE_ID IS NOT NULL)                               AS all_three,
    ROUND(100 * COUNT_IF(p.QUOTE_ID IS NOT NULL
         AND c.QUOTE_ID IS NOT NULL
         AND v.QUOTE_ID IS NOT NULL) / COUNT(*), 1)                AS all_three_pct
FROM      DEFAQTO_DB.RAW.STC_QUOTES              q
LEFT JOIN DEFAQTO_DB.RAW.STC_PERSONAL_ATTRIBUTES p USING (QUOTE_ID)
LEFT JOIN DEFAQTO_DB.RAW.STC_COVERDETAILS        c USING (QUOTE_ID)
LEFT JOIN DEFAQTO_DB.RAW.STC_VEHICLES            v USING (QUOTE_ID);

## 2 · The column nobody has looked at

`VISITED_RESULTS` in `STC_QUOTES` says whether the shopper ever reached the results
page. Run the next cell before reading on.

In [ ]:
%%sql -r dataframe_11
-- Did providers respond? Or did the shopper leave before they were asked?
WITH q AS (
  SELECT q.QUOTE_ID, q.VISITED_RESULTS,
         MAX(IFF(r.QUOTE_ID IS NOT NULL, 1, 0)) AS got_a_rate
  FROM      DEFAQTO_DB.RAW.STC_QUOTES q
  LEFT JOIN DEFAQTO_DB.RAW.STC_RATES  r USING (QUOTE_ID)
  GROUP BY 1, 2
)
SELECT VISITED_RESULTS,
       COUNT(*)                                    AS quotes,
       SUM(got_a_rate)                             AS got_a_rate,
       ROUND(100 * SUM(got_a_rate) / COUNT(*), 2)  AS pct
FROM q GROUP BY VISITED_RESULTS ORDER BY VISITED_RESULTS DESC;

### What that result means

**96% of quotes that reach the results page get a price. 0.05% of those that don't.**

So the biggest hole in the funnel is not providers failing to respond — they respond
almost every time they are asked. It is that **47% of shoppers leave before they are
asked at all**: 278,535 journeys of 589,796.

That distinction matters commercially. It is a journey problem Defaqto owns, not a
provider performance problem — and a provider should never be shown a drop-off that
was never theirs.

Nothing in SalesDB can see this today.

## 3 · Build the silver layer

Seven dynamic tables. Each one has a single job, and the cell above it says in one
sentence what you get from it.

**Why dynamic tables rather than a scheduled job.** You declare the result you want
and how stale it may become; Snowflake works out what changed and does the smallest
amount of work to catch up. No orchestration, no dependency ordering by hand, and
`TARGET_LAG = DOWNSTREAM` means a table refreshes only when something that depends
on it actually needs it.

### Prefer to have this written for you?

Paste this into Cortex Code instead of running the cells:

> Look at the seven tables in `DEFAQTO_DB.RAW` and build me a silver layer in my own
> schema. I want one row per quote with the customer's age, vehicle and cover
> attached; rates with a cheapest-to-dearest rank inside each quote; clicks tied to
> the rate that was chosen; sales with usable dates; a crosswalk resolving provider
> IDs across the quote and sales systems, which share no IDs at all; and one hub
> table at quote × provider flagging priced, clicked and sold. Use dynamic tables,
> set the refresh mode explicitly so a fallback to full refresh fails loudly rather
> than quietly, and tell me which of my joins would silently drop rows.

Then compare what it gives you with what follows. Argue with it where you disagree.

### `SILVER_PROVIDER`

**One row per provider ID per system, tying the two systems together — 253 IDs that
turn out to be only 73 real companies.**

The quote system and the sales system share **zero** provider IDs, so without this
crosswalk no cross-system join is possible at all. `Sterling` on the quote side is
`Sterling Insurance` on the sales side, so matching on name alone drops it.

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE DYNAMIC TABLE SILVER_PROVIDER
  TARGET_LAG   = DOWNSTREAM
  WAREHOUSE    = COMPUTE_WH
  REFRESH_MODE = INCREMENTAL
  COMMENT = 'One row per provider ID per system, tying the two systems together - 253 IDs that turn out to be only 73 real companies.'
AS
WITH src AS (
  SELECT 'STC'   AS SOURCE_SYSTEM, PROVIDER_ID, PROVIDER_NAME FROM DEFAQTO_DB.RAW.STC_RATES
  UNION
  SELECT 'STC',                    PROVIDER_ID, PROVIDER_NAME FROM DEFAQTO_DB.RAW.STC_CLICKS
  UNION
  SELECT 'SALES',                  PROVIDER_ID, PROVIDER_NAME FROM DEFAQTO_DB.RAW.SALESDB_SALESEVENTS
),
keyed AS (
  SELECT SOURCE_SYSTEM, PROVIDER_ID, PROVIDER_NAME,
         TRIM(REGEXP_REPLACE(UPPER(TRIM(PROVIDER_NAME)),
              ' (INSURANCE|INSURER|LTD|LIMITED|UK|GROUP)$', '')) AS PROVIDER_KEY
  FROM src
  WHERE PROVIDER_NAME IS NOT NULL
),
label AS (
  SELECT PROVIDER_KEY,
         COALESCE(MIN(IFF(SOURCE_SYSTEM = 'STC', PROVIDER_NAME, NULL)),
                  MIN(PROVIDER_NAME)) AS PROVIDER_DISPLAY
  FROM keyed GROUP BY PROVIDER_KEY
)
SELECT k.SOURCE_SYSTEM, k.PROVIDER_ID, k.PROVIDER_NAME AS PROVIDER_NAME_RAW,
       k.PROVIDER_KEY, l.PROVIDER_DISPLAY
FROM keyed k JOIN label l USING (PROVIDER_KEY);

-- Proof: the same company under several IDs and spellings.
SELECT PROVIDER_KEY, PROVIDER_DISPLAY,
       COUNT_IF(SOURCE_SYSTEM = 'STC')   AS stc_ids,
       COUNT_IF(SOURCE_SYSTEM = 'SALES') AS sales_ids,
       LISTAGG(DISTINCT PROVIDER_NAME_RAW, ' | ') AS spellings
FROM SILVER_PROVIDER
GROUP BY 1, 2 HAVING COUNT_IF(SOURCE_SYSTEM = 'STC') > 0 ORDER BY 1;

### `SILVER_QUOTES`

**Every quote request with the shopper's age, postcode, job, car and cover on one
row — so you can finally ask who was quoting.**

This is the join Ben says they cannot make: *"the product database has got all of
this extra information, but it's not connected to the sale."* Note the `LEFT` joins —
46% of quotes have no attributes at all and must survive, not vanish.

In [ ]:
%%sql -r dataframe_13
CREATE OR REPLACE DYNAMIC TABLE SILVER_QUOTES
  TARGET_LAG   = DOWNSTREAM
  WAREHOUSE    = COMPUTE_WH
  REFRESH_MODE = INCREMENTAL
  COMMENT = 'Every quote request with the shopper''s age, postcode, job, car and cover on one row - so you can finally ask who was quoting.'
AS
SELECT
    q.QUOTE_ID,
    TO_DATE(q.CREATED)                    AS QUOTE_DATE,
    q.CREATED                             AS CREATED_AT,
    q.AFFILIATE_ID,
    q.ORIGIN_ID,
    q.MOBILE,
    q.VISITED_RESULTS,
    q.VALID,
    q.INVALID_REASON,

    p.AGE_YEARS                           AS DRIVER_AGE,
    p.OUTWARD_POSTCODE,
    p.OCCUPATION_NAME,
    p.TYPE_OF_BUSINESS_NAME,
    p.LENGTH_LICENCE_HELD,

    c.REASON_FOR_COVER_LABEL              AS COVER_REASON,
    c.TYPE_OF_COVER_LABEL                 AS COVER_UNIT,
    c.COVER_LENGTH_VALUE,
    c.START_DATE                          AS COVER_START_DATE,

    v.VEHICLE_MAKE,
    v.VEHICLE_MODEL,
    v.VEHICLE_MODEL_YEAR,
    v.VEHICLE_FUEL,
    v.VEHICLE_TRANSMISSION,
    v.VEHICLE_VALUE,

    -- Cohort bands defined ONCE here, so every downstream table and both
    -- dashboards agree on what "under 7 years" means.
    CASE WHEN v.VEHICLE_MODEL_YEAR IS NULL    THEN 'unknown'
         WHEN 2026 - v.VEHICLE_MODEL_YEAR < 7 THEN 'under 7 years'
         ELSE '7 years or older' END          AS VEHICLE_AGE_BAND,
    CASE WHEN p.AGE_YEARS IS NULL THEN 'unknown'
         WHEN p.AGE_YEARS < 25    THEN '17-24'
         WHEN p.AGE_YEARS < 35    THEN '25-34'
         WHEN p.AGE_YEARS < 50    THEN '35-49'
         WHEN p.AGE_YEARS < 65    THEN '50-64'
         ELSE '65+' END                       AS DRIVER_AGE_BAND,
    CASE WHEN c.QUOTE_ID IS NULL              THEN 'unknown'
         WHEN c.TYPE_OF_COVER_LABEL = 'Hours' THEN 'hours'
         WHEN c.COVER_LENGTH_VALUE IS NULL    THEN 'unknown'
         WHEN c.COVER_LENGTH_VALUE <= 3       THEN '1-3 days'
         WHEN c.COVER_LENGTH_VALUE <= 7       THEN '4-7 days'
         ELSE '8-28 days' END                 AS COVER_LENGTH_BAND,

    -- 46% of quotes reach no product database at all. Flagging that is the
    -- honest answer to "how much of this can you actually analyse?"
    IFF(p.QUOTE_ID IS NOT NULL AND c.QUOTE_ID IS NOT NULL
        AND v.QUOTE_ID IS NOT NULL, TRUE, FALSE) AS IS_FULLY_ATTRIBUTED

FROM      DEFAQTO_DB.RAW.STC_QUOTES              q
LEFT JOIN DEFAQTO_DB.RAW.STC_PERSONAL_ATTRIBUTES p USING (QUOTE_ID)
LEFT JOIN DEFAQTO_DB.RAW.STC_COVERDETAILS        c USING (QUOTE_ID)
LEFT JOIN DEFAQTO_DB.RAW.STC_VEHICLES            v USING (QUOTE_ID);

In [ ]:
%%sql -r dataframe_26
select * from SILVER_QUOTES;

### `SILVER_RATES`

**Every price a provider returned, ranked cheapest to dearest within each quote — so
you can see where each provider landed on the page.**

`PRICE_RANK` is the single strongest predictor of a click: rank 1 wins about 3.4× as
often as rank 2. No provider can see their own rank today.

In [ ]:
%%sql -r dataframe_14
CREATE OR REPLACE DYNAMIC TABLE SILVER_RATES
  TARGET_LAG   = DOWNSTREAM
  WAREHOUSE    = COMPUTE_WH
  REFRESH_MODE = INCREMENTAL
  COMMENT = 'Every price a provider returned, ranked cheapest to dearest within each quote - so you can see where each provider landed on the page.'
AS
SELECT
    r.RATE_ID, r.QUOTE_ID, r.PROVIDER_ID, r.PROVIDER_NAME,
    px.PROVIDER_KEY, px.PROVIDER_DISPLAY,
    r.PRODUCT_NAME, r.PRICE, r.VOLUNTARY_EXCESS, r.COMPULSORY_EXCESS,
    r.STATUS, r.DECLINE_REASON, r.DATE_TIME AS RATED_AT,
    -- STATUS 3 (declined) still carries PRICE 0, so a plain AVG(PRICE) is wrong.
    IFF(r.STATUS = 1 AND r.PRICE > 0, TRUE, FALSE) AS IS_PRICED,
    -- Where this price sat on the results page.
    IFF(r.STATUS = 1 AND r.PRICE > 0,
        RANK() OVER (PARTITION BY r.QUOTE_ID
                     ORDER BY IFF(r.STATUS = 1 AND r.PRICE > 0, r.PRICE, NULL)),
        NULL)                                      AS PRICE_RANK
FROM      DEFAQTO_DB.RAW.STC_RATES r
LEFT JOIN SILVER_PROVIDER px
       ON px.SOURCE_SYSTEM = 'STC' AND px.PROVIDER_ID = r.PROVIDER_ID;

### `SILVER_CLICKS`

**Which exact price the shopper clicked — what they chose, not just that they
clicked.**

44,583 rows are flagged duplicate clicks. Count them as separate click-outs and you
overstate engagement by about 23%.

In [ ]:
%%sql -r dataframe_15
CREATE OR REPLACE DYNAMIC TABLE SILVER_CLICKS
  TARGET_LAG   = DOWNSTREAM
  WAREHOUSE    = COMPUTE_WH
  REFRESH_MODE = INCREMENTAL
  COMMENT = 'Which exact price the shopper clicked - what they chose, not just that they clicked.'
AS
SELECT
    k.CLICK_ID, k.QUOTE_ID, k.RATE_ID, k.PROVIDER_ID, k.PROVIDER_NAME,
    px.PROVIDER_KEY, px.PROVIDER_DISPLAY,
    k.DATE_TIME AS CLICKED_AT, k.VALID, k.INVALID_REASON,
    IFF(k.VALID = 1, TRUE, FALSE) AS IS_GENUINE_CLICK
FROM      DEFAQTO_DB.RAW.STC_CLICKS k
LEFT JOIN SILVER_PROVIDER px
       ON px.SOURCE_SYSTEM = 'STC' AND px.PROVIDER_ID = k.PROVIDER_ID;

### `SILVER_SALES`

**Every sale and cancellation with real dates and tidy provider names, ready to join
back to the quote it came from.**

`IS_SHORT_TERM_CAR` is the guard rail. `QUOTE_ID` is unique only *within* a product
type — join sales to quotes without it and you pick up 10,032 collision rows and
inflate GWP by 54%.

In [ ]:
%%sql -r dataframe_16
CREATE OR REPLACE DYNAMIC TABLE SILVER_SALES
  TARGET_LAG   = DOWNSTREAM
  WAREHOUSE    = COMPUTE_WH
  REFRESH_MODE = INCREMENTAL
  COMMENT = 'Every sale and cancellation with real dates and tidy provider names, ready to join back to the quote it came from.'
AS
SELECT
    s.QUOTE_ID,
    TO_DATE(s.SALE_DATE)     AS SALE_DATE,
    s.DATE_TIME              AS SOLD_AT,
    s.AFFILIATE_ID, s.AFFILIATE_NAME,
    s.PROVIDER_ID, s.PROVIDER_NAME,
    px.PROVIDER_KEY, px.PROVIDER_DISPLAY,
    s.PRODUCTTYPE_ID, s.PRODUCT_TYPE_NAME, s.PRODUCT_NAME, s.PREMIUM_TYPE,
    s.CANCELLATION, s.GWP, s.COMMISSION, s.SALE_COUNT, s.CANC_COUNT,
    IFF(s.PRODUCTTYPE_ID = 11, TRUE, FALSE) AS IS_SHORT_TERM_CAR
FROM      DEFAQTO_DB.RAW.SALESDB_SALESEVENTS s
LEFT JOIN SILVER_PROVIDER px
       ON px.SOURCE_SYSTEM = 'SALES' AND px.PROVIDER_ID = s.PROVIDER_ID;

### `SILVER_QUOTE_PROVIDER` — the hub

**One row per quote-and-provider pair: did that provider price it, get clicked, win
the sale — the backbone of every conversion number.**

This is the grain that stops a quote shown to five providers being counted five
times. Three inputs, three outputs. Materialised rather than left as a view because
every funnel number in the dashboards is computed here.

In [ ]:
%%sql -r dataframe_17
CREATE OR REPLACE DYNAMIC TABLE SILVER_QUOTE_PROVIDER
  TARGET_LAG   = DOWNSTREAM
  WAREHOUSE    = COMPUTE_WH
  REFRESH_MODE = INCREMENTAL
  COMMENT = 'One row per quote-and-provider pair: did that provider price it, get clicked, win the sale - the backbone of every conversion number.'
AS
WITH pairs AS (
  SELECT QUOTE_ID, PROVIDER_KEY FROM SILVER_RATES  WHERE PROVIDER_KEY IS NOT NULL
  UNION
  SELECT QUOTE_ID, PROVIDER_KEY FROM SILVER_CLICKS WHERE PROVIDER_KEY IS NOT NULL
  UNION
  SELECT QUOTE_ID, PROVIDER_KEY FROM SILVER_SALES
   WHERE PROVIDER_KEY IS NOT NULL AND IS_SHORT_TERM_CAR
),
rate_agg AS (
  SELECT QUOTE_ID, PROVIDER_KEY,
         COUNT(*)                         AS RATES_RETURNED,
         COUNT_IF(IS_PRICED)              AS RATES_PRICED,
         MIN(IFF(IS_PRICED, PRICE, NULL)) AS BEST_PRICE,
         MIN(PRICE_RANK)                  AS BEST_PRICE_RANK
  FROM SILVER_RATES WHERE PROVIDER_KEY IS NOT NULL GROUP BY 1, 2
),
click_agg AS (
  SELECT QUOTE_ID, PROVIDER_KEY, COUNT_IF(IS_GENUINE_CLICK) AS GENUINE_CLICKS
  FROM SILVER_CLICKS WHERE PROVIDER_KEY IS NOT NULL GROUP BY 1, 2
),
sale_agg AS (
  SELECT QUOTE_ID, PROVIDER_KEY,
         SUM(GWP) AS GWP, SUM(COMMISSION) AS COMMISSION,
         SUM(SALE_COUNT) AS SALE_COUNT, SUM(CANC_COUNT) AS CANC_COUNT
  FROM SILVER_SALES
  WHERE PROVIDER_KEY IS NOT NULL AND IS_SHORT_TERM_CAR GROUP BY 1, 2
)
SELECT
    p.QUOTE_ID, p.PROVIDER_KEY,
    NVL(r.RATES_RETURNED, 0)                       AS RATES_RETURNED,
    NVL(r.RATES_PRICED, 0)                         AS RATES_PRICED,
    r.BEST_PRICE, r.BEST_PRICE_RANK,
    NVL(c.GENUINE_CLICKS, 0)                       AS GENUINE_CLICKS,
    NVL(s.GWP, 0)                                  AS GWP,
    NVL(s.COMMISSION, 0)                           AS COMMISSION,
    IFF(NVL(r.RATES_PRICED, 0) > 0, TRUE, FALSE)   AS WAS_PRICED,
    IFF(NVL(c.GENUINE_CLICKS, 0) > 0, TRUE, FALSE) AS WAS_CLICKED,
    IFF(s.QUOTE_ID IS NOT NULL, TRUE, FALSE)       AS WAS_SOLD
FROM      pairs     p
LEFT JOIN rate_agg  r USING (QUOTE_ID, PROVIDER_KEY)
LEFT JOIN click_agg c USING (QUOTE_ID, PROVIDER_KEY)
LEFT JOIN sale_agg  s USING (QUOTE_ID, PROVIDER_KEY);

### `SILVER_SALESDB_AGGREGATE`

**daily sales summary by provider, partner and product — rebuilt so it
refreshes itself instead of running on a 1am cron.**

This is Mike's own SQL, unchanged in meaning. It produces 26,535 rows, and a row-for-row
comparison against the original query shows **zero difference in either direction**.

Two things worth noticing. `COUNT(DISTINCT ...)` refreshes **incrementally** — commonly
assumed to force a full refresh, and it does not. And `DISTINCT_QUOTES` is **not
additive**: summing it across rows overstates the true distinct count by about 2.0% (208,384 against 204,311),
which is the cleanest argument there is for a semantic view.

In [ ]:
%%sql -r dataframe_18
CREATE OR REPLACE DYNAMIC TABLE SILVER_SALESDB_AGGREGATE
  TARGET_LAG   = '1 hour'
  WAREHOUSE    = COMPUTE_WH
  REFRESH_MODE = INCREMENTAL
  COMMENT = 'Mike''s daily sales summary by provider, partner and product - rebuilt so it refreshes itself instead of running on a 1am cron.'
AS
SELECT
    SALE_DATE, AFFILIATE_ID, AFFILIATE_NAME,
    PROVIDER_ID, PROVIDER_NAME,
    PRODUCTTYPE_ID, PRODUCT_TYPE_NAME, CANCELLATION,
    COUNT(*)                 AS ROW_COUNT,
    COUNT(DISTINCT QUOTE_ID) AS DISTINCT_QUOTES,
    SUM(GWP)                 AS TOTAL_GWP,
    SUM(COMMISSION)          AS TOTAL_COMMISSION
FROM SILVER_SALES
GROUP BY SALE_DATE, AFFILIATE_ID, AFFILIATE_NAME, PROVIDER_ID, PROVIDER_NAME,
         PRODUCTTYPE_ID, PRODUCT_TYPE_NAME, CANCELLATION;

## 4 · Build the gold layer

Three tables. Each one answers a question somebody actually asked.

### `GOLD_FUNNEL_DAILY`

**How many shoppers cleared each step each day and where you lose them — including
the 47% who leave before seeing a single price.**

Carries `AFFILIATE_ID` as well as the date, so this one table serves both the insurer
view and the comparison-site view under two different row access policies. That is
why there is no separate PCW table.

In [ ]:
%%sql -r dataframe_19
CREATE OR REPLACE DYNAMIC TABLE GOLD_FUNNEL_DAILY
  TARGET_LAG   = '1 hour'
  WAREHOUSE    = COMPUTE_WH
  REFRESH_MODE = INCREMENTAL
  COMMENT = 'How many shoppers cleared each step each day and where you lose them - including the 47% who leave before seeing a single price.'
AS
WITH q AS (
  SELECT s.QUOTE_ID, s.QUOTE_DATE, s.AFFILIATE_ID, s.VISITED_RESULTS,
         MAX(IFF(h.RATES_RETURNED > 0, 1, 0)) AS got_rate,
         MAX(IFF(h.WAS_PRICED,        1, 0)) AS got_price,
         MAX(IFF(h.WAS_CLICKED,       1, 0)) AS clicked,
         MAX(IFF(h.WAS_SOLD,          1, 0)) AS sold
  FROM      SILVER_QUOTES         s
  LEFT JOIN SILVER_QUOTE_PROVIDER h USING (QUOTE_ID)
  GROUP BY 1, 2, 3, 4
)
SELECT
    QUOTE_DATE,
    AFFILIATE_ID,
    COUNT(*)                                        AS QUOTES_STARTED,
    COUNT_IF(VISITED_RESULTS = 1)                   AS REACHED_RESULTS,
    COUNT_IF(VISITED_RESULTS = 1 AND got_rate  = 1) AS RECEIVED_A_RATE,
    COUNT_IF(VISITED_RESULTS = 1 AND got_price = 1) AS RECEIVED_A_PRICE,
    COUNT_IF(clicked = 1)                           AS CLICKED_OUT,
    COUNT_IF(sold    = 1)                           AS CONVERTED,
    -- The headline: everything lost before a provider is even asked.
    COUNT(*) - COUNT_IF(VISITED_RESULTS = 1)        AS ABANDONED_BEFORE_RESULTS
FROM q
GROUP BY QUOTE_DATE, AFFILIATE_ID;

### `GOLD_PROVIDER_DAILY`

**Each insurer's own daily scorecard: quotes shown, clicks, sales, premium and
average price position.**

The screen Ben says insurers log in for: *"if I was insurer Ray, I'd see all the
places I appear and daily how well they're performing."* `AVG_PRICE_RANK` and
`TIMES_CHEAPEST` give market context without naming a competitor — which is what
makes it safe to show a partner.

In [ ]:
%%sql -r dataframe_20
CREATE OR REPLACE DYNAMIC TABLE GOLD_PROVIDER_DAILY
  TARGET_LAG   = '1 hour'
  WAREHOUSE    = COMPUTE_WH
  REFRESH_MODE = INCREMENTAL
  COMMENT = 'Each insurer''s own daily scorecard: quotes shown, clicks, sales, premium and average price position.'
AS
SELECT
    s.QUOTE_DATE,
    h.PROVIDER_KEY,
    s.AFFILIATE_ID,
    COUNT(*)                        AS QUOTES_APPEARED_IN,
    COUNT_IF(h.WAS_PRICED)          AS QUOTES_PRICED,
    COUNT_IF(h.WAS_CLICKED)         AS CLICKS,
    COUNT_IF(h.WAS_SOLD)            AS SALES,
    SUM(h.GWP)                      AS GWP,
    SUM(h.COMMISSION)               AS COMMISSION,
    AVG(h.BEST_PRICE_RANK)          AS AVG_PRICE_RANK,
    COUNT_IF(h.BEST_PRICE_RANK = 1) AS TIMES_CHEAPEST
FROM SILVER_QUOTE_PROVIDER h
JOIN SILVER_QUOTES         s USING (QUOTE_ID)
GROUP BY s.QUOTE_DATE, h.PROVIDER_KEY, s.AFFILIATE_ID;

### `GOLD_COHORT_CONVERSION`

**Which customer types each provider converts well or badly — cars under seven years
old versus older, and so on.**

The commercial question this answers: do insurers convert customers with newer cars
better than customers with older cars, and if so should pricing differ by cohort?

This table is the only reason the demographic join is worth doing.

In [ ]:
%%sql -r dataframe_21
CREATE OR REPLACE DYNAMIC TABLE GOLD_COHORT_CONVERSION
  TARGET_LAG   = '1 hour'
  WAREHOUSE    = COMPUTE_WH
  REFRESH_MODE = INCREMENTAL
  COMMENT = 'Which customer types each provider converts well or badly - cars under seven years old versus older, and so on.'
AS
SELECT
    h.PROVIDER_KEY,
    s.VEHICLE_AGE_BAND,
    s.DRIVER_AGE_BAND,
    s.COVER_LENGTH_BAND,
    NVL(s.COVER_REASON, 'unknown') AS COVER_REASON,
    COUNT(*)                       AS QUOTES_APPEARED_IN,
    COUNT_IF(h.WAS_PRICED)         AS QUOTES_PRICED,
    COUNT_IF(h.WAS_CLICKED)        AS CLICKS,
    COUNT_IF(h.WAS_SOLD)           AS SALES,
    SUM(h.GWP)                     AS GWP,
    AVG(h.BEST_PRICE)              AS AVG_BEST_PRICE,
    AVG(h.BEST_PRICE_RANK)         AS AVG_PRICE_RANK,
    IFF(s.VEHICLE_AGE_BAND  <> 'unknown'
    AND s.DRIVER_AGE_BAND   <> 'unknown'
    AND s.COVER_LENGTH_BAND <> 'unknown', TRUE, FALSE) AS IS_FULLY_ATTRIBUTED
FROM SILVER_QUOTE_PROVIDER h
JOIN SILVER_QUOTES         s USING (QUOTE_ID)
GROUP BY h.PROVIDER_KEY, s.VEHICLE_AGE_BAND, s.DRIVER_AGE_BAND,
         s.COVER_LENGTH_BAND, NVL(s.COVER_REASON, 'unknown'),
         IFF(s.VEHICLE_AGE_BAND  <> 'unknown'
         AND s.DRIVER_AGE_BAND   <> 'unknown'
         AND s.COVER_LENGTH_BAND <> 'unknown', TRUE, FALSE);

## 5 · Prove it works

Three checks. If any of them surprises you, that is the interesting part of the day.

In [ ]:
%%sql -r dataframe_22
-- Every table should say INCREMENTAL with an EMPTY refresh_mode_reason.
-- A reason means Snowflake fell back to full refresh and told you why.
-- REFRESH_MODE exists only on SHOW DYNAMIC TABLES, not in INFORMATION_SCHEMA.
SHOW DYNAMIC TABLES IN SCHEMA IDENTIFIER($my_schema);

In [ ]:
%%sql -r dataframe_23
-- refresh_mode_reason comes back NULL (not an empty string) when nothing fell
-- back, so NVL it before comparing or this column reads blank.
SELECT "name"                                                        AS table_name,
       "rows"                                                        AS n_rows,
       "target_lag"                                                  AS target_lag,
       "refresh_mode"                                                AS refresh_mode,
       IFF(NVL("refresh_mode_reason", '') = '', 'fully incremental',
           "refresh_mode_reason")                                    AS fell_back_because
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY "name";

In [ ]:
%%sql -r dataframe_25
-- Do some insurers convert newer cars much better than older ones?
-- Now answerable. Watch which providers go the OTHER way.
SELECT
    PROVIDER_KEY AS provider,
    ROUND(100 * SUM(IFF(VEHICLE_AGE_BAND = 'under 7 years',    CLICKS, 0))
              / NULLIF(SUM(IFF(VEHICLE_AGE_BAND = 'under 7 years',    QUOTES_PRICED, 0)), 0), 1) AS under_7_pct,
    ROUND(100 * SUM(IFF(VEHICLE_AGE_BAND = '7 years or older', CLICKS, 0))
              / NULLIF(SUM(IFF(VEHICLE_AGE_BAND = '7 years or older', QUOTES_PRICED, 0)), 0), 1) AS over_7_pct,
    ROUND((SUM(IFF(VEHICLE_AGE_BAND = 'under 7 years', CLICKS, 0))
           / NULLIF(SUM(IFF(VEHICLE_AGE_BAND = 'under 7 years', QUOTES_PRICED, 0)), 0))
        / NULLIF(SUM(IFF(VEHICLE_AGE_BAND = '7 years or older', CLICKS, 0))
           / NULLIF(SUM(IFF(VEHICLE_AGE_BAND = '7 years or older', QUOTES_PRICED, 0)), 0), 0), 2) AS lift
FROM GOLD_COHORT_CONVERSION
WHERE VEHICLE_AGE_BAND <> 'unknown'
GROUP BY PROVIDER_KEY
HAVING SUM(QUOTES_PRICED) > 5000
ORDER BY lift DESC;

## 6 · Look at the pipeline in Snowsight

The SQL above tells you the state. The UI shows you the shape, and it is what you
would actually use on a Monday morning.

**See the graph**

1. Left nav → **Data** → **Databases**
2. `DEFAQTO_DB` → your `TRANSFORMED_<alias>` schema → **Dynamic Tables**
3. Click **`GOLD_FUNNEL_DAILY`**
4. Open the **Graph** tab

You should see it reaching back through the hub to the four silver tables and on to
the seven landed tables. Ten nodes, and you declared no dependencies anywhere —
Snowflake worked the order out from the SQL.

**See the refresh history**

5. Same table, **Refresh History** tab
6. Note the **rows inserted / deleted** per refresh, not rows scanned

This is where incremental refresh proves itself: a table of 589,796 rows refreshing
by touching only what changed.

**Watch it react**

7. Right-click your schema → **Create** → nothing. Instead run
   `ALTER DYNAMIC TABLE GOLD_FUNNEL_DAILY REFRESH;` and reload the Refresh History
   tab. Refreshing gold pulls the `DOWNSTREAM` silver tables with it — one command,
   whole pipeline.

**What to look for**

- `TARGET_LAG = DOWNSTREAM` on silver means those tables refresh only when gold
  needs them. Nothing runs on a timer for its own sake.
- A `refresh_mode_reason` appearing later means something in the SQL changed and
  Snowflake fell back to full refresh. That is the number to watch in production.

## Done

Ten dynamic tables. Seven landed tables in, two dashboards out, and the
pre-aggregated `SalesDB_Aggregate` file replaced by something that maintains itself.

**What you can now answer that SalesDB cannot:**

- Where in the journey shoppers are lost — including the 47% who never see a price
- Which customers convert for which provider, and which do not
- Where each provider sat on the results page when it won or lost

**Next:** notebook 02 runs the same transformations as a dbt project, and notebook 03
puts a semantic view on top so you can ask these questions in English.

---

*Ketki Kothe · Solution Engineer, Snowflake · `Snowflake Solution Engineering`*
*Aggregation logic supplied by the Defaqto data lead. Sample data is synthetic.*